In [1]:
import logging
import os
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Suppress Git warnings from MLflow
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

import mlflow
import mlflow.sklearn

# Configure logging per project standards
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------------
# STEP 1: Data Preparation
# -------------------------------------------------------------------------
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
df["date_time"] = pd.to_datetime(df["date_time"])

df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

weather_dummies = pd.get_dummies(
    df["weather_main"], prefix="weather", drop_first=True
)
df = pd.concat([df, weather_dummies], axis=1)

feature_cols = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
] + list(weather_dummies.columns)

X = df[feature_cols]
y = df["traffic_volume"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------------
# STEP 2: MLflow Experiment & Version Tracking
# -------------------------------------------------------------------------
mlflow.set_experiment("Metro_Traffic_Demand_Prediction")
logger.info("MLflow Experiment set to: Metro_Traffic_Demand_Prediction")

# Model Version 1.0 (Baseline Linear Model)
with mlflow.start_run(run_name="v1.0_Ridge_Baseline"):
    reg_v1 = Ridge(alpha=1.0)
    reg_v1.fit(X_train_scaled, y_train)
    preds_v1 = reg_v1.predict(X_test_scaled)

    mae_v1 = mean_absolute_error(y_test, preds_v1)
    r2_v1 = r2_score(y_test, preds_v1)

    mlflow.log_param("model_version", "v1.0")
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_metric("mae", mae_v1)
    mlflow.log_metric("r2", r2_v1)

    mlflow.sklearn.log_model(
        sk_model=reg_v1,
        name="ridge_v1_model",
        serialization_format="cloudpickle",
    )
    logger.info(f"Logged Model v1.0 -> MAE: {mae_v1:.2f}, R2: {r2_v1:.4f}")

# Model Version 2.0 (Ensemble Tree Model)
with mlflow.start_run(run_name="v2.0_RandomForest_Ensemble"):
    reg_v2 = RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
    )
    reg_v2.fit(X_train, y_train)
    preds_v2 = reg_v2.predict(X_test)

    mae_v2 = mean_absolute_error(y_test, preds_v2)
    r2_v2 = r2_score(y_test, preds_v2)

    mlflow.log_param("model_version", "v2.0")
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_metric("mae", mae_v2)
    mlflow.log_metric("r2", r2_v2)

    mlflow.sklearn.log_model(
        sk_model=reg_v2,
        name="rf_v2_model",
        serialization_format="cloudpickle",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )
    logger.info(f"Logged Model v2.0 -> MAE: {mae_v2:.2f}, R2: {r2_v2:.4f}")

# Display Summary Table
print("\n" + "=" * 70)
print("TASK 6: MLOPS MODEL VERSIONING & EXPERIMENT TRACKING SUMMARY")
print("=" * 70)
summary_df = pd.DataFrame(
    {
        "Version": ["v1.0", "v2.0"],
        "Model Type": ["Ridge Regression", "Random Forest Regressor"],
        "Parameters Logged": ["alpha=1.0", "n_estimators=100, max_depth=12"],
        "MAE (vehicles/hr)": [round(mae_v1, 2), round(mae_v2, 2)],
        "R-Squared (R²)": [round(r2_v1, 4), round(r2_v2, 4)],
        "Deployment Stage": ["Archived (Baseline)", "Promoted to Staging/Production"],
    }
)
print(summary_df.to_string(index=False))

2026/10/05 18:37:35 INFO mlflow.tracking.fluent: Experiment with name 'Metro_Traffic_Demand_Prediction' does not exist. Creating a new experiment.


2026-10-05 18:37:35,078 - INFO - MLflow Experiment set to: Metro_Traffic_Demand_Prediction


2026/10/05 18:37:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-10-05 18:38:02,772 - INFO - Logged Model v1.0 -> MAE: 823.00, R2: 0.7199


2026/10/05 18:38:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-10-05 18:38:10,384 - INFO - Logged Model v2.0 -> MAE: 259.68, R2: 0.9461

TASK 6: MLOPS MODEL VERSIONING & EXPERIMENT TRACKING SUMMARY
Version              Model Type              Parameters Logged  MAE (vehicles/hr)  R-Squared (R²)               Deployment Stage
   v1.0        Ridge Regression                      alpha=1.0             823.00          0.7199            Archived (Baseline)
   v2.0 Random Forest Regressor n_estimators=100, max_depth=12             259.68          0.9461 Promoted to Staging/Production
